# SSS-TT Demo Inference

This notebook demonstrates how to:
1. Load a pretrained SSS-TT checkpoint
2. Run inference on a video clip
3. Visualise attention maps and pain predictions


In [ ]:
import sys, os
sys.path.insert(0, '..')

import torch
import numpy as np
import matplotlib.pyplot as plt

from src.models.sss_tt import build_sss_tt
from src.data.preprocessing import RetinaFacePreprocessor
from src.evaluation.visualization import (
    plot_attention_maps, plot_confusion_matrix, plot_robustness_curves
)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

## 1. Load Model

In [ ]:
# Load pretrained model
checkpoint_path = '../checkpoints/sss-tt/fold1/best.pth'

if os.path.exists(checkpoint_path):
    ckpt = torch.load(checkpoint_path, map_location='cpu')
    model = build_sss_tt(ckpt.get('config', {})).to(device)
    model.load_state_dict(ckpt['model'], strict=False)
    print(f'Loaded checkpoint: epoch {ckpt["epoch"]}')
else:
    # Demo with random weights
    print('No checkpoint found — using random weights for demo')
    model = build_sss_tt({
        'T': 8, 'embed_dim': 128, 'vit_depth': 2,
        'vit_heads': 4, 'tcn_layers': 2, 'modalities': []
    }).to(device)

model.eval()
n_params = sum(p.numel() for p in model.parameters())
print(f'Parameters: {n_params:,}')

## 2. Run Inference on a Video

In [ ]:
# Replace with your actual video path
VIDEO_PATH = '/data/icope/clips/subject_001/clip_001.mp4'

preprocessor = RetinaFacePreprocessor(target_size=224)

if os.path.exists(VIDEO_PATH):
    video_tensor = preprocessor.process_video(VIDEO_PATH, T=32)
else:
    # Demo with random tensor
    print('No video found — using synthetic random frames')
    video_tensor = torch.randn(8, 3, 224, 224)

print(f'Video shape: {video_tensor.shape}')

with torch.no_grad():
    out = model(video_tensor.unsqueeze(0).to(device))

print('\n--- Prediction ---')
print(f'Pain Level:   {out["pred"].item()} / 3')
print(f'Class Probs:  {out["class_probs"].squeeze().tolist()}')
labels = ['No Pain', 'Mild', 'Moderate', 'Severe']
print(f'Label:        {labels[out["pred"].item()]}')

## 3. Confidence Estimation (MC Dropout)

In [ ]:
raw_model = model.module if hasattr(model, 'module') else model

with torch.no_grad():
    result = raw_model.predict_with_confidence(
        video_tensor.unsqueeze(0).to(device),
        mc_passes=10
    )

pred = result['pred'].item()
conf = result['confidence'].item()
alert = result['alert_level'].item()
alert_map = {0: 'OK', 1: 'REVIEW', 2: 'HIGH ALERT'}

print(f'Prediction:    {labels[pred]} (Level {pred})')
print(f'Confidence:    {conf:.1%}')
print(f'Alert Level:   {alert_map[alert]}')

## 4. Visualise Attention Maps

In [ ]:
# Simulate attention for visualization
# In production, extract from model forward pass with return_attn=True
spatial_attn = np.random.dirichlet(np.ones(196))
temporal_attn = np.abs(np.random.randn(32))
temporal_attn = temporal_attn / temporal_attn.max()

fig = plot_attention_maps(
    spatial_attn, temporal_attn,
    pain_level=pred
)
plt.show()

## 5. Sample Confusion Matrix (from precomputed results)

In [ ]:
# Paper Table II results
cm = np.array([
    [132,  11,   4,   3],   # Level 0
    [ 12, 138,   7,   3],   # Level 1
    [  5,   9, 141,   5],   # Level 2
    [  3,   4,   8, 135],   # Level 3
])

fig = plot_confusion_matrix(cm, normalize=True)
plt.suptitle('SSS-TT Confusion Matrix (Test Set, Fold 1)', y=1.02)
plt.show()
print(f'Overall accuracy: {np.trace(cm)/cm.sum()*100:.1f}%')